In [1]:
# Cell 1 — Import Required Libraries

import json
import ollama
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
# Cell 2 — Load and Validate Cleaned Data

df = pd.read_csv(
    "../data/processed/customer_support_tickets_clean.csv"
)

print(f"Duplicated tickets: {df.duplicated().sum()}")

invalid_type = (
    df["Ticket Type"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    f"Invalid or empty Ticket Type values: {invalid_type}"
)

invalid_description = (
    df["Ticket Description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    f"Invalid or empty Ticket Description values: "
    f"{invalid_description}"
)

Duplicated tickets: 0
Invalid or empty Ticket Type values: 0
Invalid or empty Ticket Description values: 0


In [3]:
# Cell 3 — Recreate the ML Train/Test Pipeline

X = df["Ticket Description"]
y = df["Ticket Type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression()

model.fit(
    X_train_tfidf,
    y_train
)

print(f"Training tickets: {len(X_train)}")
print(f"Test tickets: {len(X_test)}")
print(f"TF-IDF features: {X_train_tfidf.shape[1]}")

Training tickets: 6775
Test tickets: 1694
TF-IDF features: 6290


In [4]:
# Cell 4 — Recreate the Day 30 Controlled Sample

def generate_prompt(ticket_description):
    prompt = f"""
Analyze the following customer-support ticket.

Ticket description:
{ticket_description}

Classify the ticket using exactly one of these categories:
- Technical Issue
- Refund Request
- Cancellation Request
- Billing Inquiry
- Product Inquiry

Return the analysis as JSON using exactly these four fields:

{{
    "Summary": "A short description of the customer's problem",
    "Category": "One of the five allowed ticket categories",
    "Sentiment": "The customer's expressed sentiment",
    "Urgency": "The estimated urgency of the problem"
}}

Return only the JSON object and no additional text.
"""
    return prompt


def llm_analysis(ticket_description):
    prompt = generate_prompt(ticket_description)

    response = ollama.chat(
        model="llama3.1",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    response_text = response["message"]["content"]

    analysis = json.loads(response_text)

    return analysis


ticket_categories = [
    "Technical Issue",
    "Refund Request",
    "Cancellation Request",
    "Billing Inquiry",
    "Product Inquiry"
]


ticket_sample = pd.concat([
    df[df["Ticket Type"] == ticket_type].sample(
        5,
        random_state=42
    )
    for ticket_type in ticket_categories
]).reset_index(drop=True)


display(
    ticket_sample[
        ["Ticket Description", "Ticket Type"]
    ]
)

print("\nTickets per category:")
print(
    ticket_sample["Ticket Type"].value_counts()
)

,Ticket Description,Ticket Type
0,I'm having an issue with the {product_purchase...,Technical Issue
1,I'm having an issue with the {product_purchase...,Technical Issue
2,I'm having an issue with the {product_purchase...,Technical Issue
3,I've forgotten my password for my {product_pur...,Technical Issue
4,I'm having an issue with the {product_purchase...,Technical Issue
5,I'm facing issues logging into my {product_pur...,Refund Request
6,I'm having an issue with the {product_purchase...,Refund Request
7,I'm having an issue with the {product_purchase...,Refund Request
8,I'm unable to access my {product_purchased} ac...,Refund Request
9,I'm having an issue with the {product_purchase...,Refund Request



Tickets per category:
Ticket Type
Technical Issue         5
Refund Request          5
Cancellation Request    5
Billing Inquiry         5
Product Inquiry         5
Name: count, dtype: int64


In [5]:
# Cell 5 — Generate Logistic Regression Predictions

sample_tfidf = vectorizer.transform(
    ticket_sample["Ticket Description"]
)

ml_predictions = model.predict(
    sample_tfidf
)

print(f"ML predictions generated: {len(ml_predictions)}")
print(ml_predictions)

ML predictions generated: 25
['Technical Issue' 'Technical Issue' 'Technical Issue' 'Technical Issue'
 'Product Inquiry' 'Billing Inquiry' 'Product Inquiry' 'Refund Request'
 'Product Inquiry' 'Refund Request' 'Billing Inquiry'
 'Cancellation Request' 'Billing Inquiry' 'Cancellation Request'
 'Technical Issue' 'Billing Inquiry' 'Billing Inquiry' 'Product Inquiry'
 'Billing Inquiry' 'Billing Inquiry' 'Product Inquiry'
 'Cancellation Request' 'Billing Inquiry' 'Product Inquiry'
 'Technical Issue']


In [6]:
# Cell 6 — Generate Structured LLM Predictions

llm_responses = [
    llm_analysis(description)
    for description in ticket_sample["Ticket Description"]
]

llm_predictions = [
    response["Category"]
    for response in llm_responses
]

print(f"LLM predictions generated: {len(llm_predictions)}")
print(llm_predictions)

LLM predictions generated: 25
['Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Product Inquiry', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Product Inquiry', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue']


In [7]:
# Cell 7 — Build the ML vs. LLM Comparison DataFrame

comparison = pd.DataFrame({
    "Ticket Description":
        ticket_sample["Ticket Description"],

    "Actual Type":
        ticket_sample["Ticket Type"],

    "ML Prediction":
        ml_predictions,

    "LLM Prediction":
        llm_predictions
})


comparison["ML Correct"] = (
    comparison["Actual Type"]
    == comparison["ML Prediction"]
)


comparison["LLM Correct"] = (
    comparison["Actual Type"]
    == comparison["LLM Prediction"]
)


display(comparison)

,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct
0,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
1,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
2,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
3,I've forgotten my password for my {product_pur...,Technical Issue,Technical Issue,Technical Issue,True,True
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Product Inquiry,False,False
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,False,False
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,False,False
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,False,False
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False


In [8]:
# Cell 8 — Calculate Overall Match Rates

total_tickets = len(comparison)

ml_matches = comparison["ML Correct"].sum()
llm_matches = comparison["LLM Correct"].sum()

ml_match_rate = (
    ml_matches / total_tickets
)

llm_match_rate = (
    llm_matches / total_tickets
)


print(f"Total controlled tickets: {total_tickets}")

print(
    f"ML matches: {ml_matches}/{total_tickets} "
    f"({ml_match_rate:.2%})"
)

print(
    f"LLM matches: {llm_matches}/{total_tickets} "
    f"({llm_match_rate:.2%})"
)

Total controlled tickets: 25
ML matches: 14/25 (56.00%)
LLM matches: 4/25 (16.00%)


In [9]:
# Cell 9 — Compare Results by Ticket Type

category_comparison = (
    comparison
    .groupby("Actual Type")
    .agg(
        Total_Tickets=("Actual Type", "size"),
        ML_Matches=("ML Correct", "sum"),
        LLM_Matches=("LLM Correct", "sum")
    )
    .reset_index()
)


category_comparison["ML Match Rate"] = (
    category_comparison["ML_Matches"]
    / category_comparison["Total_Tickets"]
)


category_comparison["LLM Match Rate"] = (
    category_comparison["LLM_Matches"]
    / category_comparison["Total_Tickets"]
)


display(category_comparison)

,Actual Type,Total_Tickets,ML_Matches,LLM_Matches,ML Match Rate,LLM Match Rate
0,Billing Inquiry,5,4,0,0.8,0.0
1,Cancellation Request,5,2,0,0.4,0.0
2,Product Inquiry,5,2,0,0.4,0.0
3,Refund Request,5,2,0,0.4,0.0
4,Technical Issue,5,4,4,0.8,0.8


In [10]:
# Cell 10 — Identify ML-Only, LLM-Only,
# Both-Correct, and Both-Wrong Cases

ml_only = comparison[
    comparison["ML Correct"]
    & ~comparison["LLM Correct"]
]


llm_only = comparison[
    ~comparison["ML Correct"]
    & comparison["LLM Correct"]
]


both_correct = comparison[
    comparison["ML Correct"]
    & comparison["LLM Correct"]
]


both_wrong = comparison[
    ~comparison["ML Correct"]
    & ~comparison["LLM Correct"]
]


print(f"ML only correct: {len(ml_only)}")
print(f"LLM only correct: {len(llm_only)}")
print(f"Both correct: {len(both_correct)}")
print(f"Both wrong: {len(both_wrong)}")


print("\nML Only:")
display(ml_only)


print("\nLLM Only:")
display(llm_only)


print("\nBoth Correct:")
display(both_correct)


print("\nBoth Wrong:")
display(both_wrong)

ML only correct: 10
LLM only correct: 0
Both correct: 4
Both wrong: 11

ML Only:


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False
11,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,True,False
13,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,True,False
15,I've noticed a software bug in the {product_pu...,Billing Inquiry,Billing Inquiry,Technical Issue,True,False
16,I'm having an issue with the {product_purchase...,Billing Inquiry,Billing Inquiry,Technical Issue,True,False
18,I'm having an issue with the {product_purchase...,Billing Inquiry,Billing Inquiry,Technical Issue,True,False
19,I'm having an issue with the {product_purchase...,Billing Inquiry,Billing Inquiry,Product Inquiry,True,False
20,My {product_purchased} is making strange noise...,Product Inquiry,Product Inquiry,Technical Issue,True,False
23,I'm having an issue with the {product_purchase...,Product Inquiry,Product Inquiry,Technical Issue,True,False



LLM Only:


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct



Both Correct:


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct
0,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
1,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
2,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
3,I've forgotten my password for my {product_pur...,Technical Issue,Technical Issue,Technical Issue,True,True



Both Wrong:


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Product Inquiry,False,False
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,False,False
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,False,False
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,False,False
10,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,False,False
12,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,False,False
14,"I've recently set up my {product_purchased}, b...",Cancellation Request,Technical Issue,Technical Issue,False,False
17,I'm having an issue with the {product_purchase...,Billing Inquiry,Product Inquiry,Technical Issue,False,False
21,I'm having an issue with the {product_purchase...,Product Inquiry,Cancellation Request,Technical Issue,False,False
22,I'm having an issue with the {product_purchase...,Product Inquiry,Billing Inquiry,Technical Issue,False,False


In [11]:
# Cell 11 — Inspect Prediction Disagreements

disagreements = comparison[
    comparison["ML Prediction"]
    != comparison["LLM Prediction"]
]


print(
    f"ML and LLM disagreed on "
    f"{len(disagreements)} of "
    f"{len(comparison)} tickets."
)


display(
    disagreements[
        [
            "Ticket Description",
            "Actual Type",
            "ML Prediction",
            "LLM Prediction",
            "ML Correct",
            "LLM Correct"
        ]
    ]
)

ML and LLM disagreed on 18 of 25 tickets.


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,False,False
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,False,False
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,False,False
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False
10,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,False,False
11,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,True,False
12,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,False,False
13,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,True,False
15,I've noticed a software bug in the {product_pu...,Billing Inquiry,Billing Inquiry,Technical Issue,True,False


## Day 31 Conclusions

### Direct ML vs. LLM Comparison

On Day 31, I compared the TF-IDF + Logistic Regression classifier with Llama 3.1 using the same controlled sample of 25 customer-support tickets. The sample contained 5 tickets from each of the five ticket categories so that both approaches were evaluated on exactly the same descriptions.

### Overall Results

The Logistic Regression model matched the dataset labels for 14 of the 25 tickets, producing a 56% match rate on the controlled sample.

Llama 3.1 matched the dataset labels for 4 of the 25 tickets, producing a 16% match rate.

Therefore, the traditional ML approach matched the dataset labels more frequently than the LLM on this specific controlled sample. This result does not establish that Logistic Regression is generally better than Llama 3.1 because the experiment used only 25 tickets and evaluated agreement with the existing dataset labels.

### Results by Ticket Type

The Logistic Regression model performed differently across the five ticket types:

- Billing Inquiry: 4 of 5 matched (80%)
- Cancellation Request: 2 of 5 matched (40%)
- Product Inquiry: 2 of 5 matched (40%)
- Refund Request: 2 of 5 matched (40%)
- Technical Issue: 4 of 5 matched (80%)

Llama 3.1 matched 4 of the 5 Technical Issue labels (80%), but matched 0 of 5 tickets in each of the other four categories.

The LLM showed a strong tendency to classify tickets as Technical Issue. This behavior was especially noticeable for tickets whose dataset labels were Refund Request, Cancellation Request, Billing Inquiry, or Product Inquiry but whose descriptions described problems that could sound technical.

### Correctness Comparison

Of the 25 tickets:

- Both approaches matched the dataset label: 4
- Only Logistic Regression matched the dataset label: 10
- Only Llama 3.1 matched the dataset label: 0
- Both approaches failed to match the dataset label: 11

The two approaches therefore behaved differently even though they received the same ticket descriptions. Logistic Regression was the only correct method on 10 examples, while there were no examples in this sample where only Llama 3.1 matched the dataset label.

### Prediction Disagreements

The ML and LLM approaches produced different category predictions for 18 of the 25 tickets. This shows that the two approaches often interpreted the same descriptions differently.

A disagreement does not necessarily mean that one system produced a semantically unreasonable answer. The dataset label was used as the ground truth for calculating match rates, but some descriptions contain generic or problem-focused language that may not clearly indicate the assigned ticket category.

### Conclusion

The same-ticket comparison showed that TF-IDF + Logistic Regression matched the dataset labels substantially more often than Llama 3.1 in this 25-ticket experiment. However, the high number of disagreements and both-wrong cases suggested that model performance alone might not explain all of the errors.

The next step is to inspect these errors more closely and determine whether characteristics of the ticket descriptions, such as generic wording, template placeholders, or possible text-label ambiguity, may contribute to the classification difficulty.